# Entrenar y evaluar los cuatro modelos de Sem1

Este notebook ejecuta una corrida local reproducible para **T5S, BARTO, mBART-50 y uMT5-Base**. Usa el Gold congelado de 400 noticias: 320 train, 40 validation y 40 test. La inferencia se realiza solo sobre test y las métricas se calculan después contra sus 40 claims de referencia: METEOR, ROUGE-L Precision/Recall/F1 y BERTScore Precision/Recall/F1.

No modifica el Gold, no consulta APIs y no añade recuperación, ranking ni multiclaim. Los pesos y resultados quedan en `artifacts/runs/`, ignorado por Git.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, *cwd.parents) if (path / 'scripts' / 'run_sem1_experiment.py').exists())
GOLD_DIR = ROOT / 'artifacts' / 'sem1_20261031' / 'final'
RUN_ID = 'sem1_20261031_400_claims_v1'
RUN_DIR = ROOT / 'artifacts' / 'runs' / RUN_ID
MODELS = ['t5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es']

TRAIN = GOLD_DIR / 'train.csv'
VALIDATION = GOLD_DIR / 'validation.csv'
TEST = GOLD_DIR / 'test.csv'

for path in (TRAIN, VALIDATION, TEST):
    assert path.is_file(), f'Falta el Gold: {path}'

print({'repo': str(ROOT), 'run_dir': str(RUN_DIR), 'models': MODELS})

{'repo': '/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification', 'run_dir': '/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1', 'models': ['t5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es']}


## 1. Verificar el Gold antes de entrenar

La corrida se detiene si detecta un tamaño incorrecto, claims vacíos o fuga de `record_id`, `source_hash` o `claim_hash` entre particiones.

In [2]:
splits = {name: pd.read_csv(path) for name, path in {'train': TRAIN, 'validation': VALIDATION, 'test': TEST}.items()}
assert {name: len(frame) for name, frame in splits.items()} == {'train': 320, 'validation': 40, 'test': 40}
assert all(frame['reference_claim'].fillna('').str.strip().ne('').all() for frame in splits.values())
for column in ('record_id', 'source_hash', 'claim_hash'):
    values = [set(frame[column]) for frame in splits.values()]
    assert not (values[0] & values[1] or values[0] & values[2] or values[1] & values[2]), column

pd.DataFrame([{'split': name, 'filas': len(frame), 'claims únicos': frame['claim_hash'].nunique()} for name, frame in splits.items()])

,split,filas,claims únicos
0,train,320,320
1,validation,40,40
2,test,40,40


## 2. Verificar el dispositivo

PyTorch usa CUDA cuando está disponible, luego MPS en Apple Silicon y, si ninguno está activo, CPU. El dispositivo real queda registrado en `run_status.json` y en cada `training_manifest.json`.

In [3]:
import torch

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
{'device': device, 'torch': torch.__version__, 'mps_built': torch.backends.mps.is_built()}

{'device': 'mps', 'torch': '2.15.0', 'mps_built': True}

## 3. Ejecutar entrenamiento, inferencia y métricas

La celda es reanudable por defecto: si no existe `RUN_DIR`, inicia una corrida nueva; si ya existe, conserva los modelos con checkpoint y evaluación válidos. Para reiniciar desde cero, define otro `RUN_ID`; así no se mezclan resultados parciales.

In [4]:
RESUME = True

command = [
    sys.executable, 'scripts/run_sem1_experiment.py',
    '--train', str(TRAIN),
    '--validation', str(VALIDATION),
    '--test', str(TEST),
    '--run-dir', str(RUN_DIR),
    '--models', *MODELS,
]
if RESUME:
    command.append('--resume')

print(' '.join(command))
subprocess.run(command, cwd=ROOT, check=True)

/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/.venv/bin/python scripts/run_sem1_experiment.py --train /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/sem1_20261031/final/train.csv --validation /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/sem1_20261031/final/validation.csv --test /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/sem1_20261031/final/test.csv --run-dir /Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1 --models t5s_spanish barto_spanish mbart50_es umt5_base_es --resume


{
  "run_dir": "/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1",
  "models": [
    "t5s_spanish",
    "barto_spanish",
    "mbart50_es",
    "umt5_base_es"
  ]
}


CompletedProcess(args=['/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/.venv/bin/python', 'scripts/run_sem1_experiment.py', '--train', '/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/sem1_20261031/final/train.csv', '--validation', '/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/sem1_20261031/final/validation.csv', '--test', '/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/sem1_20261031/final/test.csv', '--run-dir', '/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1', '--models', 't5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es', '--resume'], returncode=0)

## 4. Resumen de métricas de la corrida

Esta tabla rápida muestra los F1 principales y deja disponibles los desgloses P/R. Las métricas comparan la salida con `reference_claim`; no miden por sí solas veracidad médica. El notebook `02_resultados_modelos.ipynb` genera las tablas formateadas para el informe.

In [5]:
summary_path = RUN_DIR / 'evaluation' / 'summary.csv'
required_metrics = [
    'model_display_name', 'meteor',
    'rouge_l_precision', 'rouge_l_recall', 'rouge_l_f1',
    'bertscore_precision', 'bertscore_recall', 'bertscore_f1',
]
if summary_path.is_file():
    metrics = pd.read_csv(summary_path)
    missing = sorted(set(required_metrics).difference(metrics.columns))
    if missing:
        print('La corrida usa un formato anterior. Ejecuta de nuevo la evaluación para añadir:', ', '.join(missing))
    else:
        display(
            metrics[required_metrics].rename(columns={
                'model_display_name': 'Modelo',
                'meteor': 'METEOR',
                'rouge_l_precision': 'ROUGE-L P',
                'rouge_l_recall': 'ROUGE-L R',
                'rouge_l_f1': 'ROUGE-L F1',
                'bertscore_precision': 'BERTScore P',
                'bertscore_recall': 'BERTScore R',
                'bertscore_f1': 'BERTScore F1',
            }).style.format({column: '{:.3f}' for column in required_metrics[1:]})
        )
else:
    print('Aún no existe evaluation/summary.csv. Ejecuta la celda anterior.')

,Modelo,METEOR,ROUGE-L P,ROUGE-L R,ROUGE-L F1,BERTScore P,BERTScore R,BERTScore F1
0,T5S,0.173504,0.355973,0.185653,0.220925,0.841211,0.809392,0.824626
1,BARTO,0.315000,0.351639,0.316687,0.325780,0.903427,0.896149,0.899658
2,mBART-50,0.325756,0.358986,0.332755,0.335128,0.904701,0.898072,0.901226
3,uMT5-Base,0.345111,0.397288,0.351832,0.363487,0.909671,0.901609,0.905471


## 5. Revisar el estado y continuar con las tablas

Al terminar debe aparecer `status: completed` y una fila por modelo. Luego abre `02_resultados_modelos.ipynb`, conserva el mismo `RUN_ID` y ejecuta todas sus celdas para ver las tablas.

In [6]:
status_path = RUN_DIR / 'run_status.json'
if status_path.is_file():
    status = json.loads(status_path.read_text(encoding='utf-8'))
    display(pd.DataFrame.from_dict(status['models'], orient='index'))
    print({'status': status['status'], 'device': status['device'], 'logs': str(RUN_DIR / 'logs')})
else:
    print('Aún no hay una corrida creada. Ejecuta la celda anterior.')

,status,summary
t5s_spanish,skipped_completed,"{'model_id': 't5s_spanish', 'model_display_nam..."
barto_spanish,skipped_completed,"{'model_id': 'barto_spanish', 'model_display_n..."
mbart50_es,skipped_completed,"{'model_id': 'mbart50_es', 'model_display_name..."
umt5_base_es,skipped_completed,"{'model_id': 'umt5_base_es', 'model_display_na..."


{'status': 'completed', 'device': 'mps', 'logs': '/Users/renzo/Documents/GitHub/peruvian-medical-misinformation-classification/artifacts/runs/sem1_20261031_400_claims_v1/logs'}
